<a href="https://colab.research.google.com/github/hiiamlars/analytical_flexibility_llm_reviews/blob/main/data_generation/llm_review_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Review segmentation

## Set up

In [ ]:
# @title Dependencies

# First-party installations
import os
from google.colab import drive
from IPython.display import display
import hashlib
import json

# Third-party installations
import pandas as pd
import numpy as np

In [ ]:
# @title Path definitions

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Analytical_flexibility_llm_reviews"

# Create working directory
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create DATASET_DIR
DATASET_DIR = os.path.join(WORKING_DIR, "llm_reviews/generated")
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Ensured dataset directory exists at: {DATASET_DIR}.")

# Define and create the RAW_DIR
RAW_DIR = os.path.join(WORKING_DIR, "llm_reviews/segmented")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RAW_DIR}.")

# Define the full result path (final Parquet output)
RESULTS_PATH = os.path.join(RAW_DIR, "results.parquet")
# Define the full result path for intermediate JSONL storage
RESULTS_JSON_PATH = os.path.join(RAW_DIR, "results.jsonl")

In [ ]:
# @title Setup definitions

# Define review section of interest
TARGET_SECTION = "weaknesses"

# Define target columns
REVIEW_COL_ORIGINAL = 'llm_review'
REVIEW_COL_EXTRACTED_WEAKNESS = 'extracted_weaknesses'
REVIEW_COL_EXTRACTED_RATING = 'extracted_scores'
REVIEW_COL_SEGMENTS = 'segmented_comment'

# Define id columns
CONFIG_UNI_ID = "config_group"
UNI_ID = "segment_hash"

# Define config columns for grouping hash
config_columns = [
    "model",
    "temperature",
    "reasoning_effort",
    "chain_of_thought",
    "note_taking"
]

# Define the desired order of columns
COL_ORDER = [
    UNI_ID,
    CONFIG_UNI_ID,
    'paper_id',
    'run_signature',
    'model',
    'temperature',
    'reasoning_effort',
    'chain_of_thought',
    'note_taking',
    'llm_notes',
    'llm_review',
    REVIEW_COL_EXTRACTED_WEAKNESS,
    REVIEW_COL_EXTRACTED_RATING,
    REVIEW_COL_SEGMENTS,
    'correctness_rating',
    'technical_novelty_significance_rating',
    'empirical_novelty_significance_rating',
    'generation_successful'
]

# Define keys for numerical ratings
RATING_KEYS = [
    "correctness_rating",
    "technical_novelty_significance_rating",
    "empirical_novelty_significance_rating"
]

# Define error string for failed review generation
REVIEW_GENERATION_FAILURE = "ERROR: REVIEW GENERATION NOT SUCCESSFUL"

# Define error string for failed review comment segmentation
EXTRACTION_FAILURE = "ERROR: SEGMENT EXTRACTION FAILED"

In [ ]:
# @title Data import

# Read dataset
df = pd.read_parquet(os.path.join(DATASET_DIR, "results.parquet"))

# Check and display
display(df.shape)
display(df.head(3))

In [ ]:
# @title Configuration group identifier

# Create an informative identifier for each configuration group
df[CONFIG_UNI_ID] = df.apply(lambda row: ' | '.join([f'{col}: {row[col]}' for col in config_columns]), axis=1)

# Check and display grouping identifier(s)
display(df.shape)
display(df.head(3))

## Define

In [ ]:
# @title Extraction logic

def extract_weaknesses(json_str):
    """
    Parses the json_str and extracts the TARGET_SECTION section.
    Returns a list of strings, or a list containing an error string if parsing fails or the section is missing.
    """
    # Handle specific error strings first
    if json_str in [REVIEW_GENERATION_FAILURE]:
        return [json_str]

    try:
        # Load columnn data into JSON
        review_data = json.loads(json_str)

        # Isolate the TARGET_SECTION
        target_section = review_data.get(TARGET_SECTION)

        # If target_section is None or empty, return the specific error string
        if not target_section:
            return [EXTRACTION_FAILURE]

        # Return the whole section
        return target_section

    except Exception as e:
        # Print a hardcoded error message for debugging
        print(f"Warning: Failed to isolate weaknesses. Error: {e}. Review string: {json_str[:100]}...")
        return [EXTRACTION_FAILURE]

def extract_ratings(json_str, rating_keys):
    """
    Parses the json_str and extracts numerical ratings for specified keys.
    Returns a dictionary of ratings, or the error string for missing keys.
    If the input is an error string, all ratings will be that error string.
    """
    ratings = {}
    # Handle specific error strings first
    if json_str == REVIEW_GENERATION_FAILURE:
        for key in rating_keys:
            ratings[key] = REVIEW_GENERATION_FAILURE # Assign the error string
        return ratings

    try:
        review_data = json.loads(json_str)
        for key in rating_keys:
            ratings[key] = review_data.get(key) # Assign actual scores
    except Exception as e:
        print(f"Warning: Failed to extract ratings. Error: {e}. Review string: {json_str[:100]}...")
        for key in rating_keys:
            ratings[key] = EXTRACTION_FAILURE # Assign EXTRACTION_FAILURE for parsing errors
    return ratings

## Run

In [ ]:
# @title Extraction

# Apply the extraction function for weaknesses
df[REVIEW_COL_EXTRACTED_WEAKNESS] = df[REVIEW_COL_ORIGINAL].apply(extract_weaknesses)

# Apply the extraction function for ratings
extracted_ratings_series = df[REVIEW_COL_ORIGINAL].apply(lambda x: extract_ratings(x, RATING_KEYS))

# Add the full dictionary of ratings as a new column
df[REVIEW_COL_EXTRACTED_RATING] = extracted_ratings_series

# Expand the dictionary of ratings into separate columns
for key in RATING_KEYS:
    df[key] = extracted_ratings_series.apply(lambda x: x.get(key))

# Check and display extracted section(s) and new rating columns
display(df.shape)
display(df.head())


In [ ]:
# @title Success metric

# Add metric column for later analysis
# Check if the list in REVIEW_COL_EXTRACTED_WEAKNESS is NOT equal to the specific error lists.
df['generation_successful'] = ~df[REVIEW_COL_EXTRACTED_WEAKNESS].apply(lambda x: x == [REVIEW_GENERATION_FAILURE] or x == [EXTRACTION_FAILURE])

## Transformation

In [ ]:
# @title Long-format

# Create a copy of the extracted section column
df[REVIEW_COL_SEGMENTS] = df[REVIEW_COL_EXTRACTED_WEAKNESS]

# Convert the copied column into long-format
df = df.explode(REVIEW_COL_SEGMENTS)

# Display and check segmented comment(s)
display(df.shape)
display(df.head(3))

In [ ]:
# @title Unique identifiers

# Get all columns as basis for unique identifier (hash)
col_for_hash = df.columns.tolist()

# Create a unique identifier (hash)
df[UNI_ID] = df[col_for_hash].astype(str).agg(''.join, axis=1).apply(lambda x: hashlib.sha1(x.encode('utf-8')).hexdigest()[:12])

# Display and check unique identifier(s)
display(df.shape)
display(df.head(3))

In [ ]:
# @title Reorder columns

# Reindex the dataset to apply the new column order
df = df.reindex(columns=COL_ORDER)

# Display and check the reordered dataset
display(df.shape)
display(df.head(3))

In [ ]:
# @title Results

# Convert all columns to strings (ensures the dataset can be saved to .parquet)
df = df.map(lambda x: str(x) if x is not None else None)

# Save the long-format dataset to JSONL
df.to_json(RESULTS_JSON_PATH, orient='records', lines=True)
print(f"Dataset saved to JSONL at: {RESULTS_JSON_PATH}")

# Save the long-format Dataset to Parquet
df.to_parquet(RESULTS_PATH, index=False)
print(f"Dataset saved to Parquet at: {RESULTS_PATH}")